# Part 2 — GPUs on NRP

Two ways to use a GPU on NRP:

1. **Interactively**, in a JupyterHub session started with a GPU (this notebook, if you picked one).
2. **As a batch job**: describe the run in a YAML file, and the cluster finds a free GPU, runs it and releases the GPU when it exits.

> 📘 [GPU pods](https://nrp.ai/documentation/userdocs/running/gpu-pods/) · [What's free right now](https://nrp.ai/viz/resources/) · [Getting started with kubectl](https://nrp.ai/documentation/userdocs/start/getting-started/)

## 1. What GPUs are there?

The quickest view is the live resource page, [nrp.ai/viz/resources](https://nrp.ai/viz/resources/). From a terminal, the GPU model is a label on every node:

In [ ]:
%%bash
# GPUs on NRP, counted from the node labels
if ! nodes=$(kubectl get nodes --no-headers -o custom-columns='N:.metadata.labels.nvidia\.com/gpu\.count,MODEL:.metadata.labels.nvidia\.com/gpu\.product' 2>/dev/null); then
  echo "This account can't list nodes; use https://nrp.ai/viz/resources/ instead."; exit 0
fi
echo "$nodes" | awk '$1 != "<none>" { n[$2] += $1; t += $1 }
  END { printf "%d GPUs. The 15 most common:\n\n", t; fflush(); for (m in n) printf "%5d  %s\n", n[m], m | "sort -rn | head -15" }'

## 2. A GPU in this session

If you started this server with a GPU, `nvidia-smi` shows it. (No GPU? The cells below say so and skip.)

In [ ]:
%%bash
nvidia-smi || echo "No GPU in this session."

The same matrix multiply on the CPU and on the GPU. For a GPU, a 4096 × 4096 multiply is a few milliseconds of work.

In [ ]:
%%bash
# multiply two 4096 x 4096 matrices on the CPU, then on the GPU
python - <<'EOF'
import time, torch

def tflops(device, n=4096, reps=5):
    a, b = torch.randn(n, n, device=device), torch.randn(n, n, device=device)
    a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return 2 * n**3 * reps / (time.time() - t) / 1e12

cpu = tflops("cpu", reps=2)
print(f"CPU: {cpu:6.2f} TFLOP/s")
if torch.cuda.is_available():
    gpu = tflops("cuda")
    print(f"GPU: {gpu:6.1f} TFLOP/s  on {torch.cuda.get_device_name(0)}  ({gpu / cpu:.0f}x)")
else:
    print("No GPU in this session.")
EOF

A GPU in a notebook is great for exploring and debugging. But it stays yours **until you stop the server**, even while you're at lunch. For real runs, use a job.

## 3. The same work as a batch job

`../yamls/gpu-job.yaml` asks for one GPU with more than 20 GB of memory, of the Ampere generation or newer, then trains a small network on handwritten digits (MNIST) and exits. The part that matters:

In [ ]:
%%bash
# the job, up to the training script
sed -n '/^apiVersion/,/^        command:/p' ../yamls/gpu-job.yaml | sed '$d'

Each person's job needs its own name, so we put your hub username into it.

In [ ]:
%%bash
# your job's name, from your hub username; saved for the next cells
ME=$(echo "${JUPYTERHUB_USER:-me}" | cut -d@ -f1 | tr '[:upper:]' '[:lower:]' | sed -E 's/[^a-z0-9]+/-/g; s/^-+//; s/-+$//' | cut -c1-30)
printf 'ME=%s\nJOB=gpu-train-%s\nNS=nrp-training-k8s\n' "$ME" "$ME" > job.env
cat job.env

### Submit it

In [ ]:
%%bash
source job.env
sed "s/<username>/$ME/" ../yamls/gpu-job.yaml | kubectl apply -f -

### Watch it

**Pending** means the scheduler is waiting for a free GPU that matches. On a busy cluster that can take a few minutes, and the first run on a node also downloads the container image. This cell prints each change of state, then streams the job's output.

In [ ]:
%%bash
source job.env
last=""; start=$SECONDS
while true; do
  phase=$(kubectl get pods -n "$NS" -l job-name="$JOB" -o jsonpath='{.items[0].status.phase} {.items[0].spec.nodeName}' 2>/dev/null | xargs)
  if [ "$phase" != "$last" ]; then
    printf '%5d s  %s\n' $((SECONDS - start)) "${phase:-waiting for the pod}"
    last=$phase
  fi
  case "$phase" in Running*|Succeeded*|Failed*) break ;; esac
  sleep 3
done
kubectl logs -f "job/$JOB" -n "$NS"

Stuck in Pending? `describe` says why, usually that no GPU of the kind you asked for is free:

In [ ]:
%%bash
source job.env
kubectl describe pod -n "$NS" -l job-name="$JOB" | sed -n '/Events:/,$p'

### Clean up

The job deletes itself an hour after it finishes (`ttlSecondsAfterFinished`), but it's good practice to clean up:

In [ ]:
%%bash
source job.env
kubectl delete job "$JOB" -n "$NS"

## 4. In your own namespace

To run this outside the training hub:

1. Get a namespace: PIs, faculty, staff and postdocs request one at [get-started.nrp-nautilus.io](https://get-started.nrp-nautilus.io/); students are added by their PI.
2. Install `kubectl` and the NRP kubeconfig: [Getting started](https://nrp.ai/documentation/userdocs/start/getting-started/).
3. Change `namespace:` in the YAML, then `kubectl apply -f gpu-job.yaml`.

**Going further:** keep data on a persistent volume rather than the pod's own disk; run a sweep as many one-GPU jobs; and see the [CMS HATS training](https://training.nrp-nautilus.io/cms-hats/) for a physics neural network trained in GPU jobs with storage and experiment tracking.